<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>


## **6. Understanding Training**

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 6.1 Introduction

So far we have always started from a finished model: [05.04](05.04_GettingStarted_with_Raspberry_Pi.ipynb) ran one on the Raspberry Pi, and [05.05](05.05_Introduction_to_LLMs.ipynb) showed how it turns probabilities into text. This notebook fills the gap in between: **where do the weights come from?**

In the middle of the notebook, a tiny language model goes from random noise to readable sentences in about half a minute on your laptop. It is built from parts you already know: token embeddings ([05.02](05.02_Understanding_Word_Embedding.ipynb)), attention ([05.03](05.03_Understanding_Attention.ipynb)) and next-token probabilities ([05.05](05.05_Introduction_to_LLMs.ipynb)). Training fills in their values, and it works the same way at every scale: a frontier model is about a million times larger than ours, but it is trained with the same loop.

Allow about **45 minutes**, plus time for the questions. By the end, you should be able to explain what the loss measures, how gradient descent and the learning rate reduce it, what a bigger model buys and what it costs, and why a freshly trained model is not yet an assistant.

#### 6.1.1 Environment Setup

Everything runs on your laptop with `torch`. Only Section 6.6 asks the Pi one question.

<div class="alert alert-block alert-success">

If a library is missing, use the installation instructions in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb). Select that Python environment as this notebook's kernel.
</div>

### 6.2 The Objective: Measuring Surprise

Training needs a single number to improve, and for language models that number is quite simple. The model reads text and, at every position, assigns a probability to each possible next character. The **loss** measures how surprised the model is when the true next character arrives:

$$
\text{loss} \;=\; -\log_2\, p(\text{true next character})
$$

Measured this way, the unit of surprise is the **bit**, the same unit you met in [`01.5_Entropy.ipynb`](../01_Data_and_Signals/01.5_Entropy.ipynb). A model that assigns the true character a probability of 0.5 pays exactly one bit, the surprise of one fair coin flip. Averaged over a whole text, the loss is the number of bits the model pays per character, and training tries to push this average down.

Our model later in this notebook chooses among 28 characters. Three imaginary models show how the loss behaves before we train anything:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

VOCAB_SIZE = 28   # the character set our tiny model will use later

scenarios = {
    "clueless: spreads probability evenly": 1 / VOCAB_SIZE,
    "confident and right: p = 0.9":         0.9,
    "confident and wrong: p = 0.001":       0.001,
}

surprise_bits = {name: -np.log2(p) for name, p in scenarios.items()}

fig, ax = plt.subplots(figsize=(10, 3.2))
bars = ax.barh(list(surprise_bits), list(surprise_bits.values()), height=0.55, color="#4c78a8")
for bar, bits in zip(bars, surprise_bits.values()):
    ax.text(bits + 0.15, bar.get_y() + bar.get_height() / 2, f"{bits:.2f} bits", va="center")

ax.invert_yaxis()
ax.set_xlabel("surprise when the true next character arrives (bits)")
ax.set_xlim(0, 11.5)
ax.set_title("The loss rewards confidence only when it is justified", loc="left")
fig.tight_layout()
plt.show()

### 6.3 Learning: Gradient Descent

The chart shows two things. The clueless model pays $\log_2 28 \approx 4.8$ bits per character. This number comes back later as the starting point of our training run. And being confidently wrong costs far more than spreading the probability evenly: the loss punishes certainty that is not justified.

How does the loss go down? A model is a large set of adjustable numbers, the weights. **Gradient descent** improves them step by step: predict a batch of text, measure the surprise, compute for every weight the direction that would have reduced it, and nudge every weight a small step that way. Then repeat, millions of times. You saw this loop shape attention patterns in [`05.03_Understanding_Attention.ipynb`](./05.03_Understanding_Attention.ipynb). Here we look at one setting of the loop itself.

That setting is the **learning rate**: how large each step is. The plot below shows the simplest possible case, one weight and a loss shaped like a valley, with three learning rates. Real models do the same in millions of dimensions at once.

In [ ]:
learning_rates = [(0.02, "too small", "#4c78a8"),
                  (0.25, "well chosen", "#1b9e77"),
                  (1.05, "too large", "#d95f02")]

w_axis = np.linspace(-4.5, 4.5, 200)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)

for ax, (lr, label, color) in zip(axes, learning_rates):
    w, path = 3.0, [3.0]
    for _ in range(12):
        w = w - lr * 2 * w          # gradient of loss(w) = w^2 is 2w
        path.append(w)
    path = np.array(path)

    ax.plot(w_axis, w_axis ** 2, color="#bbbbbb", linewidth=1.2, zorder=1)
    ax.plot(path, path ** 2, color=color, linewidth=1.2, marker="o",
            markersize=5, zorder=3)
    ax.plot(path[0], path[0] ** 2, marker="o", markersize=9, color=color, zorder=4)
    ax.set_title(f"{label}: lr = {lr}", loc="left", fontsize=10, color=color)
    ax.set_xlim(-4.5, 4.5)
    ax.set_ylim(-0.5, 14)
    ax.set_xlabel("the weight being adjusted")

axes[0].set_ylabel("loss")
plt.show()

Each panel starts at the same large dot and takes twelve steps. With a small learning rate the weight creeps downhill, and most of the computing time is wasted. With a well-chosen rate it reaches the bottom in a few steps. With a rate that is too large, each step overshoots and lands higher on the opposite slope: the loss grows, and the run is ruined.

At our scale a ruined run costs half a minute. A large training run occupies thousands of accelerators for months, so there a ruined run costs a great deal of money. This is why labs try out their settings on small models before they start a large run.

### 6.4 Watching a Language Model Learn

We need three things, one cell each: data, a model and the training loop.

#### 6.4.1 The data

A model can only learn what its training data contains, however long we train. To make that visible, we build the corpus ourselves from sentence templates about devices, signals and wireless links. The vocabulary is tiny and the sentences repeat on purpose. This is the idea behind TinyStories [04]: very simple data lets a very small model learn to write coherent text.

In [ ]:
import random

def build_corpus(n_sentences=4000, seed=0):
    rng = random.Random(seed)
    devices = ["laptop", "phone", "router", "sensor", "drone", "tablet", "camera", "robot"]
    links = ["wi-fi", "bluetooth", "lifi", "5g"]
    places = ["office", "kitchen", "garden", "basement", "classroom", "street"]
    templates = [
        "the {d} sends data over {l}.",
        "the {d} in the {p} connects over {l}.",
        "when the signal is weak, the {d} sends the data again.",
        "the {d} measures the signal and reports it.",
        "a strong signal means fast data for the {d}.",
        "the {d} talks to the {d2} over {l}.",
        "in the {p}, the {l} signal is strong.",
        "the {d} waits, then sends a short message over {l}.",
        "the battery of the {d} runs low, so it sends less data.",
        "the {d} listens before it talks on the shared channel.",
    ]
    sentences = []
    for _ in range(n_sentences):
        t = rng.choice(templates)
        sentences.append(t.format(d=rng.choice(devices), d2=rng.choice(devices),
                                  l=rng.choice(links), p=rng.choice(places)))
    return " ".join(sentences)


corpus = build_corpus()

# The characters are our token vocabulary. We add "?" so we can ask questions later.
chars = sorted(set(corpus) | {"?"})
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
VOCAB_SIZE = len(chars)

print(f"corpus: {len(corpus):,} characters")
print(f"vocabulary ({VOCAB_SIZE} characters): {''.join(chars)!r}")
print(f"\nbeginning of the corpus:\n  {corpus[:150]}...")

#### 6.4.2 The model

The model is a small version of a real LLM, and every part of it comes from an earlier notebook: an embedding table turns each character into a vector, a stack of transformer blocks applies attention, and a final layer turns the result into a probability for each of the 28 characters. Real LLMs have the same parts, only far larger.

We keep it at about a quarter of a million parameters so that a full training run fits into half a minute of laptop CPU. For comparison, the smallest chat model on your Pi, `qwen3.5:0.8b`, holds about three thousand times more.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(2)   # a tiny model gains nothing from more threads, and many threads stall whenever another program runs

torch.manual_seed(0)

CONTEXT = 48   # how many previous characters the model may look at

class TinyLanguageModel(nn.Module):
    def __init__(self, vocab_size, context, d_model=96, n_heads=4, n_layers=2):
        super().__init__()
        self.context = context
        # characters -> vectors (05.02)
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(context, d_model)
        # attention blocks (05.03)
        layer = nn.TransformerEncoderLayer(
            d_model, n_heads, dim_feedforward=4 * d_model,
            dropout=0.0, batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        # vectors -> a score for every character in the vocabulary
        self.head = nn.Linear(d_model, vocab_size)
        nn.init.zeros_(self.head.weight)   # start with the same score for every character
        nn.init.zeros_(self.head.bias)

    def forward(self, idx):
        T = idx.shape[1]
        h = self.token_embedding(idx) + self.position_embedding(torch.arange(T))
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T)
        h = self.blocks(h, mask=causal_mask, is_causal=True)   # may only look backwards
        return self.head(h)


@torch.no_grad()
def sample_text(model, prompt, n_chars=100, temperature=0.8, seed=1):
    """Generate text character by character, as in 05.05."""
    generator = torch.Generator().manual_seed(seed)
    model.eval()
    idx = torch.tensor([[stoi[c] for c in prompt]])
    for _ in range(n_chars):
        logits = model(idx[:, -CONTEXT:])
        probs = F.softmax(logits[0, -1] / temperature, dim=-1)
        next_char = torch.multinomial(probs, 1, generator=generator)
        idx = torch.cat([idx, next_char.view(1, 1)], dim=1)
    model.train()
    return "".join(itos[int(i)] for i in idx[0])


model = TinyLanguageModel(VOCAB_SIZE, CONTEXT)
n_parameters = sum(p.numel() for p in model.parameters())
print(f"model parameters : {n_parameters/1e3:.0f} thousand")
print(f"for comparison   : qwen3.5:0.8b has {0.8e9/n_parameters:,.0f}x more")

#### 6.4.3 The training loop

The loop below is the whole training. It repeats four steps: **draw** a random batch of text snippets, **predict** every next character, **measure** the surprise in bits, and **nudge** every weight downhill. Large models are trained with the same loop, only with more data, more weights and extra techniques that keep the training stable.

So that we can watch the model learn, the loop pauses at a few checkpoints and lets the model write text after the prompt `"the "`. Compare what it writes at step 0 with what it writes at step 150 and at the end.

In [ ]:
data = torch.tensor([stoi[c] for c in corpus], dtype=torch.long)

def get_batch(batch_size=32):
    starts = torch.randint(len(data) - CONTEXT - 1, (batch_size,))
    x = torch.stack([data[i:i + CONTEXT] for i in starts])        # what the model sees
    y = torch.stack([data[i + 1:i + CONTEXT + 1] for i in starts])  # the true next characters
    return x, y


torch.manual_seed(0)
model = TinyLanguageModel(VOCAB_SIZE, CONTEXT)   # a fresh, untrained model, also when you rerun this cell

# Before training starts, record what the model believes follows "... over w".
probe_prompt = "the router sends data over w"
with torch.no_grad():
    model.eval()
    probe_logits = model(torch.tensor([[stoi[c] for c in probe_prompt]]))
    probs_before = F.softmax(probe_logits[0, -1], dim=-1).clone()
    model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
STEPS = 500
SAMPLE_AT = {0, 15, 60, 150}
LOG2 = float(np.log(2))

import time
losses = []
start_time = time.time()

for step in range(STEPS):
    if step in SAMPLE_AT:
        print(f"--- step {step:4d} ---")
        print(f"  {sample_text(model, 'the ')}\n")

    x, y = get_batch()                                                  # 1. draw
    logits = model(x)                                                   # 2. predict
    loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1)) / LOG2   # 3. measure
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()                                                    # 4. nudge

    losses.append(float(loss.detach()))
    if step % 100 == 99:
        print(f"step {step + 1:4d}: {np.mean(losses[-50:]):.3f} bits/char   "
              f"({time.time() - start_time:.0f} s elapsed)")

print(f"\n--- step {STEPS} (final) ---")
print(f"  {sample_text(model, 'the ')}")
print(f"\ntrained {STEPS} steps in {time.time() - start_time:.0f} s on this machine")

with torch.no_grad():
    model.eval()
    probe_logits = model(torch.tensor([[stoi[c] for c in probe_prompt]]))
    probs_after = F.softmax(probe_logits[0, -1], dim=-1)
    model.train()

Read the samples from top to bottom. At step 0 the model writes noise: every character is about equally likely, at 4.8 bits of surprise each. Within a few dozen steps it learns the simplest patterns: which letters are common, that a space follows a word, how to spell `the`. Then come whole words, then word order, and the grammar settles last. Nobody planned this order. Frequent, local patterns lower the loss more per step than rare, long-range ones, so gradient descent learns them first.

Nobody told the model how to spell `bluetooth` or how a sentence is built. It picked up both from the raw text, because that lowered its surprise. Large models acquire what looks like knowledge in the same way.

#### 6.4.4 The loss curve

The samples showed the progress in words. The loss curve shows it in numbers: the surprise per character at every step.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))

smoothed = np.convolve(losses, np.ones(25) / 25, mode="valid")
ax.plot(losses, color="#1b9e77", alpha=0.25, linewidth=0.8)
ax.plot(np.arange(len(smoothed)) + 12, smoothed, color="#1b9e77", linewidth=1.8,
        label="loss (smoothed)")

ax.axhline(np.log2(VOCAB_SIZE), color="#757575", linestyle="--", linewidth=1.0)
ax.text(len(losses) * 0.55, np.log2(VOCAB_SIZE) + 0.12,
        f"clueless model: log2({VOCAB_SIZE}) = {np.log2(VOCAB_SIZE):.1f} bits", color="#757575")

for step in SAMPLE_AT:
    ax.axvline(step, color="#bbbbbb", linewidth=0.8, zorder=1)
samples = ax.secondary_xaxis("top")
samples.set_xticks(sorted(SAMPLE_AT))
samples.tick_params(labelsize=8, colors="#555555")
samples.set_xlabel("text samples printed in Section 6.4.3", fontsize=8, color="#555555")

ax.set_xlabel("training step")
ax.set_ylabel("loss (bits per character)")
ax.set_ylim(0, 5.2)
ax.set_xlim(0, len(losses))
ax.legend(loc="center right")
ax.set_title("Surprise falls as the data's patterns move into the weights", loc="left")
plt.show()

The curve starts at the clueless line from Section 6.2: before training, the network is the clueless model. The steep fall at the start is the simple patterns being learned, and the long flat tail is everything that is harder to learn. Nearly all of the progress happened in the first third of the run, and every further step bought less. Section 6.5 comes back to this shape. The faint line is the loss of each single batch. It jitters because every batch is a new random set of snippets, and the smoothed curve, an average over 25 steps, keeps a little of that jitter.

#### 6.4.5 What changed inside

What does it mean that the weights moved? In [05.05](05.05_Introduction_to_LLMs.ipynb) you looked at the next-token probabilities of the model on your Pi. Here is the same view for our model, before and after training, after the prompt `"the router sends data over w"`. In our corpus the only link that starts with `w` is `wi-fi`, so a model that has learned the data should be nearly certain that the next character is `i`.

In [ ]:
top_after = torch.argsort(probs_after, descending=True)[:6]
labels = ["space" if itos[int(i)] == " " else repr(itos[int(i)]) for i in top_after]
before_values = [float(probs_before[i]) for i in top_after]
after_values = [float(probs_after[i]) for i in top_after]

positions = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(positions - 0.18, before_values, width=0.36, color="#757575", label="before training")
ax.bar(positions + 0.18, after_values, width=0.36, color="#1b9e77", label="after training")

for x, v in zip(positions + 0.18, after_values):
    ax.text(x, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)

ax.set_xticks(positions, labels)
ax.set_ylabel("probability of the next character")
ax.set_ylim(0, 1.12)
ax.legend()
ax.set_title(f'P(next character) after the prompt "{probe_prompt}"', loc="left")
plt.show()

Before training, the probability is spread thinly across the whole alphabet. After training, nearly all of it sits on `i`. The same happened for every context in the training data. That is what training produced: not stored sentences, but next-character probabilities for every context, held in the weights.

### 6.5 A Tiny Scaling Law

Every training budget raises the same question: **what does a bigger model buy?** We can measure it on our own problem. The cell below trains four models of increasing size on the same data for the same 150 steps, and records the final loss and the training time of each. It takes well under a minute, and the growing time per model is part of the result.

In [ ]:
import pandas as pd

configurations = [
    dict(d_model=24,  n_heads=2, n_layers=1),
    dict(d_model=48,  n_heads=2, n_layers=1),
    dict(d_model=96,  n_heads=4, n_layers=2),
    dict(d_model=160, n_heads=4, n_layers=3),
]
SCALING_STEPS = 150

records = []
for config in configurations:
    torch.manual_seed(0)
    candidate = TinyLanguageModel(VOCAB_SIZE, CONTEXT, **config)
    opt = torch.optim.AdamW(candidate.parameters(), lr=3e-3)
    t0 = time.time()
    recent = []
    for step in range(SCALING_STEPS):
        x, y = get_batch()
        loss = F.cross_entropy(candidate(x).reshape(-1, VOCAB_SIZE), y.reshape(-1)) / LOG2
        opt.zero_grad(); loss.backward(); opt.step()
        recent.append(float(loss.detach()))
    records.append({
        "parameters": sum(p.numel() for p in candidate.parameters()),
        "training seconds": round(time.time() - t0, 1),
        "final loss (bits/char)": round(float(np.mean(recent[-30:])), 3),
        "d_model": config["d_model"],
    })
    print(f"d_model={config['d_model']:4d}: done in {records[-1]['training seconds']} s")

scaling = pd.DataFrame(records).set_index("d_model")
print()
print(scaling)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4), constrained_layout=True)
for ax, column, color in [(axes[0], "final loss (bits/char)", "#1b9e77"),
                          (axes[1], "training seconds", "#4c78a8")]:
    ax.plot(scaling["parameters"], scaling[column], color=color, alpha=0.4)
    ax.scatter(scaling["parameters"], scaling[column], s=80, color=color, zorder=3)
    for d_model, row in scaling.iterrows():
        ax.annotate(f"d={d_model}", (row["parameters"], row[column]),
                    textcoords="offset points", xytext=(0, 9), ha="center", fontsize=8)
    ax.set_xscale("log")
    ax.set_xlabel("parameters (log scale)")
    ax.set_ylabel(column)

axes[0].set_ylim(0, None)
axes[1].set_ylim(0, None)
axes[0].set_title("What a bigger model buys ...", loc="left")
axes[1].set_title("... and what it costs", loc="left")
plt.show()

Quality improves steeply at first and then flattens, while the cost keeps growing.

Two results carry over to real models. The **scaling laws** [01]: for real models on real data, the left curve keeps improving for a long time, smoothly and predictably, which lets labs plan training budgets years ahead. And the **compute-optimal balance** [02]: for a fixed budget, a smaller model trained on more data often beats a larger one trained on less, so the amount and quality of the training data matter as much as the size of the model. Once training is finished, the techniques of [`05.09_Making_Models_Smaller.ipynb`](./05.09_Making_Models_Smaller.ipynb) make the model small enough to deploy.

### 6.6 A Trained Model Is Not Yet an Assistant

Our model has now learned its small world well. Two more prompts show how it differs from the chat models you use: a question, and a word that its corpus never contained.

In [ ]:
print("QUESTION PROMPT")
print("  " + sample_text(model, "what is wi-fi? ", n_chars=90))
print()
print("A WORD THE TRAINING DATA NEVER CONTAINED")
print("  " + sample_text(model, "the satellite ", n_chars=90))

The model does not answer the question. It continues the text with more sentences in the style of its corpus: its training never rewarded answering, only plausible continuation. A freshly pretrained model of any size behaves this way, and such a **base model** is not yet an assistant.

The model also writes about `satellite`, a word it has never seen, without any sign of trouble. It was trained to write fluent text; whether the text is true was never part of the loss. This is the root of what is called **hallucination**, and it is why a good demo tells you little about what a model knows.

Turning a model that continues text into an assistant takes further training stages, which are much cheaper than pretraining:

| Stage | Data it consumes | Rough scale | What it teaches |
|:------|:-----------------|:------------|:----------------|
| **Pretraining** | trillions of tokens of web text, books, code | months on thousands of accelerators. Public estimates start in the tens of millions of dollars | language, facts, style: to *continue* any text |
| **Instruction tuning (SFT)** | thousands to millions of curated question-answer dialogues | days on a small cluster | to *answer* rather than continue |
| **Preference alignment (RLHF / DPO)** | human rankings of candidate answers | comparable to instruction tuning | to be helpful, harmless, and honest about uncertainty |

Section 6.4 was pretraining in miniature. Reasoning models add one more training stage, and [05.13](05.13_Experimenting_with_Prompts.ipynb) shows one at work. After training come compression for deployment ([05.09](05.09_Making_Models_Smaller.ipynb)) and the decoding settings at inference time ([05.12](05.12_Understanding_Inference_Control.ipynb)). In the project [`05.18_Build_Your_Own_Model.ipynb`](./05.18_Build_Your_Own_Model.ipynb) you build the whole pipeline yourself, at a small scale.

For contrast, here is the same question to the model on your Pi, which went through all three stages. Connect the Pi first, as in [05.04](05.04_GettingStarted_with_Raspberry_Pi.ipynb).

In [ ]:
import caspi

caspi.connect()
caspi.ask("granite4.2:3b", "what is wi-fi?", system="Answer in two sentences.", think=False)

### 6.7 Playground

The trained model is still in memory. Try your own prompts and get a feeling for what the weights contain and what they do not.

In [ ]:
# Try your own prompts (all characters must exist in the vocabulary printed in Section 6.4.1).
# Ideas: prompts from the corpus world, half-finished sentences, words the corpus
# never contained, temperature 0.1 versus 1.5.

print(sample_text(model, "when the signal is ", n_chars=100, temperature=0.8))

To go further, rerun Section 6.4.3 with more steps, or give the model a larger `d_model`.

### 6.8 Summarizing Remarks

The idea behind training is simple, but doing it at scale is enormously expensive. The loss measures surprise in bits per character, gradient descent nudges millions of weights to reduce it, and everything a model "knows" (spelling, grammar, facts, style) is patterns from the training data that ended up in the weights because learning them lowered the surprise. What is not in the data, the model cannot know, but it will still write about it fluently.

A larger model buys quality smoothly and predictably, but its cost keeps growing, so real budgets balance the size of the model against the amount of data. Pretraining alone gives a model that continues text, while instruction tuning and preference alignment turn it into an assistant.

You will probably never pretrain a model yourself as only a handful of labs can afford it. Your decisions come later: which model to use, whether to fine-tune it, and whether to give it your own documents through retrieval. [05.07](05.07_Introduction_to_Local_LLMs.ipynb) looks at choosing a model and [05.10](05.10_Local_RAG_meets_Tit-for-Tat.ipynb) at retrieval.

### 6.9 Further Reading and Literature

[01] Kaplan, J., McCandlish, S., Henighan, T., Brown, T. B., et al. (2020). **Scaling Laws for Neural Language Models.** arXiv. [Paper](https://arxiv.org/abs/2001.08361)

[02] Hoffmann, J., Borgeaud, S., Mensch, A., et al. (2022). **Training Compute-Optimal Large Language Models.** NeurIPS. [Paper](https://arxiv.org/abs/2203.15556)

[03] Ouyang, L., Wu, J., Jiang, X., et al. (2022). **Training language models to follow instructions with human feedback (InstructGPT).** NeurIPS. [Paper](https://arxiv.org/abs/2203.02155)

[04] Eldan, R., & Li, Y. (2023). **TinyStories: How Small Can Language Models Be and Still Speak Coherent English?** arXiv. [Paper](https://arxiv.org/abs/2305.07759). The idea behind our template corpus.

[05] Karpathy, A. (2023). [**Let's build GPT: from scratch, in code, spelled out.**](https://www.youtube.com/watch?v=kCc8FmEb1nY) YouTube video lecture. Our tiny model follows the spirit of this walkthrough.

[06] Sanderson, G. (3Blue1Brown). [**Gradient descent, how neural networks learn.**](https://www.youtube.com/watch?v=IHZwWFHWa-w) YouTube video lecture.

---
### 6.10 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

Use your own outputs as evidence. Copy the relevant numbers into your answer so that they remain visible after saving.

**Question 01: The first and the last loss**

Report the first loss of your training run, `losses[0]`, and the last one the loop printed. The first one matches a number computed long before training started. Which one, and why must that be so?

(answer here)

**Question 02: Three learning rates**

Describe what happened in each of the three learning-rate panels. Why is a learning rate that is too large a more expensive mistake than one that is too small?

(answer here)

**Question 03: What the model learned first**

Paste one checkpoint sample from early, middle, and late in your run. What did the model master first: letters, words, or word order? Give a reason for that order.

(answer here)

**Question 04: Where abilities come from**

Nobody told the model how to spell `bluetooth` or how to build a sentence, yet the trained model does both. Where did that come from, and what does it imply about where a large model's abilities and biases come from?

(answer here)

**Question 05: The scaling table**

Report your scaling table (parameters, seconds, final loss). Between which two sizes was the extra quality per extra second smallest? What do the compute-optimal results [02] recommend when buying quality: only a larger model, or something else?

(answer here)

**Question 06: A question to a base model**

What did our model do with the question `what is wi-fi?`, and what did the model on your Pi do? Which stage of the pipeline in Section 6.6 makes the difference?

(answer here)

**Question 07: A word it never saw**

The model wrote fluently about `the satellite`, which its corpus never contained. Explain why this happens, and what it means for judging any language model by a demonstration.

(answer here)

**Question 08: Three ways to an assistant**

Your organisation wants an assistant that knows its internal documents. Rank the three options: pretrain from scratch, fine-tune an open-weights model, or retrieval over documents (RAG). Use the table from Section 6.6 to justify your ranking.

(answer here)

**Question 09: Optional: one change of your own**

Change one thing in the pipeline (more training steps, a different temperature in the playground, a template of your own in the corpus) and report what changed in the output and why.

(answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)